# ⚡ MediaFire to Google Drive Cloud GUI Transfer
Transfer files from MediaFire directly to your Google Drive at ultra-high speed (up to 100+ MB/s) using Google Cloud servers and `aria2` 16x multi-connection acceleration.

### Step 1: Connect your Google Drive
Run this cell below, click the authorization link, and sign in. (Alternatively, click the **Files 📁** icon in the left sidebar and click the **Mount Drive** folder button).

In [ ]:
from google.colab import drive
import os

gdrive_root = "/content/drive/MyDrive" if os.path.exists("/content/drive/MyDrive") else "/content/drive/My Drive"

if os.path.ismount('/content/drive') and os.path.exists(gdrive_root):
    print(f"\033[92m[SUCCESS] Google Drive is already connected at: {gdrive_root}\033[0m")
    print("\033[94m[IMPORTANT] Ensure you are viewing the SAME Google Account on drive.google.com in your browser!\033[0m")
else:
    print("[1/2] Connecting to Google Drive...")
    try:
        drive.mount('/content/drive')
    except ValueError:
        print("\033[93m[NOTICE] Re-mounting Google Drive to refresh connection...\033[0m")
        drive.mount('/content/drive', force_remount=True)
        
    gdrive_root = "/content/drive/MyDrive" if os.path.exists("/content/drive/MyDrive") else "/content/drive/My Drive"
    if os.path.exists(gdrive_root):
        print(f"\n\033[92m[SUCCESS] Google Drive successfully mounted at: {gdrive_root}\033[0m")
        print("\033[94m[IMPORTANT] Ensure you are viewing the SAME Google Account on drive.google.com in your browser!\033[0m")
    else:
        print("\n\033[91m[WARNING] Mount completed but MyDrive directory not found. Please verify Google Drive authorization.\033[0m")


### Step 2: Install High-Speed Transfer Engine
Run this cell to set up `aria2` 16x multi-connection accelerator. This only needs to be run once per session.

In [ ]:
!apt-get update -qq && apt-get install -y aria2

### Step 3: Interactive Transfer Queue (With Movie/Subfolder Support)
Use the interactive panel below to queue multiple MediaFire links with optional folder grouping:
1. **Base GDrive Folder**: Main destination directory in `MyDrive` (default: `MediaFire_Transfers`).
2. **Movie / Subfolder Name (Optional)**: If a movie has multiple links (e.g. CD1 & CD2, or Part 1 & Part 2), enter the movie/folder name here. All links you add will automatically be grouped into that folder.
3. **Add Links One by One**: Paste each link and click **➕ Add to Queue** (or press Enter). They will be added to the queue with their assigned folder.
4. **Start Transfers**: Click **🚀 Start Transfer** to download and stream each file into its designated folder with real-time terminal output.

> **Tip:** You can enter multiple links at once for the same movie (separated by commas or newlines), or provide a path to a `.txt` file.

In [ ]:
#@title Interactive Transfer Queue Manager (MediaFire to Google Drive)
# Run this cell to open the Interactive Link Queue Manager

import os
import subprocess
import time
import sys
import re
import fcntl
import pty
import requests
import urllib.parse
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets

def get_gdrive_root():
    for candidate in ["/content/drive/MyDrive", "/content/drive/My Drive"]:
        if os.path.exists(candidate):
            return candidate
    return None

if 'mf_link_queue' not in globals():
    mf_link_queue = []

def extract_mediafire_links(text):
    if not text:
        return []
    clean_text = text.replace(',', ' ').replace('"', ' ').replace("'", ' ')
    candidates = clean_text.split()
    clean_links = []
    for token in candidates:
        token = token.strip().rstrip('.,;')
        if 'mediafire.com' in token:
            if not token.startswith('http://') and not token.startswith('https://'):
                token = 'https://' + token
            if token not in clean_links:
                clean_links.append(token)
    return clean_links

def get_mediafire_direct_url(page_url):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }
    if not page_url.startswith('http://') and not page_url.startswith('https://'):
        page_url = 'https://' + page_url
    
    try:
        res = requests.get(page_url, headers=headers, allow_redirects=False, timeout=10)
        if res.status_code in (301, 302, 303, 307, 308) and 'Location' in res.headers:
            loc = res.headers['Location']
            if 'mediafire.com' in loc:
                return loc
    except:
        pass
            
    try:
        res = requests.get(page_url, headers=headers, stream=True, timeout=10)
        ct = res.headers.get('Content-Type', '').lower()
        if 'text/html' not in ct:
            url = res.url
            res.close()
            return url
            
        html_chunks = []
        bytes_read = 0
        for chunk in res.iter_content(chunk_size=16384):
            html_chunks.append(chunk)
            bytes_read += len(chunk)
            if bytes_read > 500000:
                break
        res.close()
        html = b"".join(html_chunks).decode('utf-8', errors='ignore')
    except Exception as e:
        print(f"\033[91m[ERROR] Connection error resolving MediaFire link: {e}\033[0m")
        return None

    match = re.search(r'href=["\'](https?://download\d*\.mediafire\.com/[^"\']+)["\']', html)
    if match:
        return match.group(1)
        
    match = re.search(r'aria-label="Download file"[^>]*href=["\']([^"\']+)["\']', html)
    if match:
        return match.group(1)
        
    match = re.search(r'id="downloadButton"[^>]*href=["\']([^"\']+)["\']', html)
    if match:
        return match.group(1)
        
    match = re.search(r'class="inputpops"[^>]*href=["\']([^"\']+)["\']', html)
    if match:
        return match.group(1)
        
    return None

def sanitize_folder_name(name):
    if not name:
        return ""
    cleaned = name.strip()
    # Keep & and normal characters intact without changing to 'and'
    for bad_char in ['<', '>', ':', '"', '/', '\\', '|', '?', '*']:
        cleaned = cleaned.replace(bad_char, '_')
    cleaned = ' '.join(cleaned.split())
    return cleaned.strip(' .')

def resolve_subfolder_path(movie_val, inner_val):
    parts = []
    for raw in [movie_val, inner_val]:
        if raw:
            for segment in raw.replace('\\', '/').split('/'):
                seg_clean = sanitize_folder_name(segment)
                if seg_clean and not (seg_clean.startswith('[') and seg_clean.endswith(']')):
                    parts.append(seg_clean)
    if parts:
        return os.path.join(*parts), " / ".join(parts)
    return "", ""

def get_direct_or_mediafire_title(link):
    if not link:
        return None
    import urllib.parse, re
    parsed = urllib.parse.urlparse(link)
    path = parsed.path
    if path:
        base = os.path.basename(path.rstrip('/'))
        base = urllib.parse.unquote(base)
        clean = re.sub(r'\.[a-zA-Z0-9]{2,5}$', '', base)
        if clean and clean.lower() != 'file':
            return clean
    return None

def get_unique_filename(directory, filename):
    base, ext = os.path.splitext(filename)
    counter = 1
    new_name = filename
    while os.path.exists(os.path.join(directory, new_name)):
        new_name = f"{base} ({counter}){ext}"
        counter += 1
    return new_name

# UI Controls
# 1. Link & Auto-Name Controls
link_input = widgets.Text(
    value="",
    description="Download Link(s):",
    placeholder="Paste MediaFire or Direct Web Link (e.g. patrins, mp4, mkv, zip)...",
    style={'description_width': '145px'},
    layout=widgets.Layout(width='500px')
)

auto_name_btn = widgets.Button(
    description=" 🪄 Auto-Pull Name",
    icon="magic",
    button_style="info",
    tooltip="Auto-pull folder/movie name from the link",
    layout=widgets.Layout(width='150px', height='36px')
)

link_row = widgets.HBox([link_input, auto_name_btn], layout=widgets.Layout(margin='2px 0'))

# 2. Destination Controls
base_folder_input = widgets.Text(
    value="MediaFire_Transfers",
    description="Base GDrive Folder:",
    placeholder="Main folder in MyDrive",
    style={'description_width': '145px'},
    layout=widgets.Layout(width='655px')
)

def get_existing_gdrive_folders():
    gdrive_root = get_gdrive_root()
    if not gdrive_root:
        return []
    base_gdrive = sanitize_folder_name(base_folder_input.value.strip()) or "MediaFire_Transfers"
    base_path = os.path.join(gdrive_root, base_gdrive)
    if not os.path.exists(base_path):
        return []
    try:
        folders = [f for f in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, f)) and not f.startswith('.')]
        return sorted(folders, key=lambda s: s.lower())
    except Exception:
        return []

def get_existing_inner_subfolders(movie_name):
    gdrive_root = get_gdrive_root()
    if not gdrive_root or not movie_name:
        return []
    base_gdrive = sanitize_folder_name(base_folder_input.value.strip()) or "MediaFire_Transfers"
    movie_path = os.path.join(gdrive_root, base_gdrive, movie_name)
    if not os.path.exists(movie_path) or not os.path.isdir(movie_path):
        return []
    try:
        subdirs = [f for f in os.listdir(movie_path) if os.path.isdir(os.path.join(movie_path, f)) and not f.startswith('.')]
        return sorted(subdirs, key=lambda s: s.lower())
    except Exception:
        return []

existing_folders = get_existing_gdrive_folders()

folder_dropdown = widgets.Dropdown(
    options=["[ ➕ New Movie / Type Below ]"] + existing_folders,
    value="[ ➕ New Movie / Type Below ]",
    description="Select Movie Folder:",
    style={'description_width': '145px'},
    layout=widgets.Layout(width='500px')
)

refresh_folders_btn = widgets.Button(
    description=" Refresh",
    icon="refresh",
    button_style="",
    tooltip="Scan Google Drive for existing movie folders",
    layout=widgets.Layout(width='150px', height='36px')
)

subfolder_input = widgets.Text(
    value="",
    description="Movie Folder Name:",
    placeholder="Select from dropdown above or type a new movie name...",
    style={'description_width': '145px'},
    layout=widgets.Layout(width='655px')
)

inner_dropdown = widgets.Dropdown(
    options=["[ No Inside Folder (Save in Movie Root) ]", "[ ➕ Create New Inside Folder ]"],
    value="[ No Inside Folder (Save in Movie Root) ]",
    description="Select Inside Folder:",
    style={'description_width': '145px'},
    layout=widgets.Layout(width='500px')
)

inner_subfolder_input = widgets.Text(
    value="",
    description="Inside Folder Name:",
    placeholder="e.g. Dialogue, Songs, 4K Clips (Optional - leave blank if not needed)",
    style={'description_width': '145px'},
    layout=widgets.Layout(width='655px')
)

def on_folder_dropdown_changed(change):
    selected_movie = change['new']
    if selected_movie and not selected_movie.startswith('['):
        subfolder_input.value = selected_movie
        inners = get_existing_inner_subfolders(selected_movie)
        inner_dropdown.options = ["[ No Inside Folder (Save in Movie Root) ]", "[ ➕ Create New Inside Folder ]"] + inners
        inner_dropdown.value = "[ No Inside Folder (Save in Movie Root) ]"
        inner_subfolder_input.value = ""
    elif selected_movie == "[ ➕ New Movie / Type Below ]":
        subfolder_input.value = ""
        inner_dropdown.options = ["[ No Inside Folder (Save in Movie Root) ]", "[ ➕ Create New Inside Folder ]"]
        inner_dropdown.value = "[ No Inside Folder (Save in Movie Root) ]"
        inner_subfolder_input.value = ""

folder_dropdown.observe(on_folder_dropdown_changed, names='value')

def on_subfolder_input_changed(change):
    val = change.get('new', '').strip()
    if 'http://' in val or 'https://' in val:
        if not link_input.value.strip():
            link_input.value = val
        title = get_direct_or_mediafire_title(val)
        if title:
            subfolder_input.value = title
            folder_dropdown.value = "[ ➕ New Movie / Type Below ]"
            with status_output:
                clear_output()
                print(f"\033[92m✨ Auto-detected Movie Folder Name: '{title}'\033[0m")
            return
    if val:
        inners = get_existing_inner_subfolders(val)
        current_opts = list(inner_dropdown.options)
        new_opts = ["[ No Inside Folder (Save in Movie Root) ]", "[ ➕ Create New Inside Folder ]"] + inners
        if new_opts != current_opts:
            inner_dropdown.options = new_opts

subfolder_input.observe(on_subfolder_input_changed, names='value')

def on_inner_dropdown_changed(change):
    selected_inner = change['new']
    if selected_inner and not selected_inner.startswith('['):
        inner_subfolder_input.value = selected_inner
    elif selected_inner == "[ No Inside Folder (Save in Movie Root) ]":
        inner_subfolder_input.value = ""
    elif selected_inner == "[ ➕ Create New Inside Folder ]":
        inner_subfolder_input.value = ""

inner_dropdown.observe(on_inner_dropdown_changed, names='value')

def on_auto_name_clicked(b=None):
    raw_text = link_input.value.strip()
    sub_text = subfolder_input.value.strip()

    if ('http://' in sub_text or 'https://' in sub_text) and not ('http://' in raw_text or 'https://' in raw_text):
        raw_text = sub_text
        if not link_input.value.strip():
            link_input.value = sub_text
        subfolder_input.value = ""

    if not raw_text:
        with status_output:
            clear_output()
            print("\033[93m[TIP] Paste a link into 'Download Link(s)' first, then click 'Auto-Pull Name'.\033[0m")
        return

    first_link = raw_text.splitlines()[0].strip() if '\n' in raw_text else raw_text.split(',')[0].strip()
    with status_output:
        clear_output()
        print(f"\033[96m[INFO] Extracting folder/movie name from link...\033[0m")
    title = get_direct_or_mediafire_title(first_link)
    if title:
        subfolder_input.value = title
        folder_dropdown.value = "[ ➕ New Movie / Type Below ]"
        with status_output:
            clear_output()
            print(f"\033[92m✨ Auto-detected Movie Folder Name: '{title}'\033[0m")
    else:
        with status_output:
            clear_output()
            print("\033[93m[NOTE] Could not auto-detect folder name from this link. You can type it manually.\033[0m")

auto_name_btn.on_click(on_auto_name_clicked)

def on_link_input_changed(change):
    raw_text = change.get('new', '').strip()
    if raw_text and not subfolder_input.value.strip() and (folder_dropdown.value == "[ ➕ New Movie / Type Below ]" or not folder_dropdown.value):
        first_link = raw_text.splitlines()[0].strip() if '\n' in raw_text else raw_text.split(',')[0].strip()
        title = get_direct_or_mediafire_title(first_link)
        if title and not subfolder_input.value.strip():
            subfolder_input.value = title
            with status_output:
                clear_output()
                print(f"\033[92m✨ Auto-detected Movie Folder Name from link: '{title}'\033[0m")

link_input.observe(on_link_input_changed, names='value')

def on_refresh_folders_clicked(b=None):
    folders = get_existing_gdrive_folders()
    current = folder_dropdown.value
    folder_dropdown.options = ["[ ➕ New Movie / Type Below ]"] + folders
    if current in folder_dropdown.options:
        folder_dropdown.value = current
    elif subfolder_input.value in folder_dropdown.options:
        folder_dropdown.value = subfolder_input.value
    if subfolder_input.value:
        inners = get_existing_inner_subfolders(subfolder_input.value.strip())
        inner_dropdown.options = ["[ No Inside Folder (Save in Movie Root) ]", "[ ➕ Create New Inside Folder ]"] + inners
    with status_output:
        clear_output()
        if folders:
            print(f"\033[92m[INFO] Scanned Google Drive: Found {len(folders)} existing movie folder(s).\033[0m")
        else:
            print(f"\033[93m[INFO] No existing movie folders found in '{base_folder_input.value.strip()}' yet.\033[0m")

refresh_folders_btn.on_click(on_refresh_folders_clicked)

def on_base_folder_changed(change):
    folders = get_existing_gdrive_folders()
    folder_dropdown.options = ["[ ➕ New Movie / Type Below ]"] + folders
    folder_dropdown.value = "[ ➕ New Movie / Type Below ]"

base_folder_input.observe(on_base_folder_changed, names='value')

link_input = widgets.Text(
    value="",
    description="MediaFire Link(s):",
    placeholder="Paste MediaFire link here and click 'Add to Queue' (or press Enter)...",
    style={'description_width': '140px'},
    layout=widgets.Layout(width='655px')
)

add_btn = widgets.Button(
    description=" Add to Queue",
    icon="plus",
    button_style="info",
    layout=widgets.Layout(width='140px', height='36px')
)

remove_btn = widgets.Button(
    description=" Remove Last",
    icon="minus",
    button_style="warning",
    layout=widgets.Layout(width='130px', height='36px')
)

clear_btn = widgets.Button(
    description=" Clear Queue",
    icon="trash",
    button_style="danger",
    layout=widgets.Layout(width='130px', height='36px')
)

reset_btn = widgets.Button(
    description=" Reset Status",
    icon="refresh",
    button_style="",
    layout=widgets.Layout(width='130px', height='36px')
)

start_btn = widgets.Button(
    description=" Start Transfer",
    icon="play",
    button_style="success",
    layout=widgets.Layout(width='170px', height='40px')
)

queue_output = widgets.Output()
status_output = widgets.Output()

def render_queue():
    with queue_output:
        clear_output(wait=True)
        if not mf_link_queue:
            display(HTML('''
            <div style="border: 1px dashed #ced4da; border-radius: 8px; padding: 16px; margin: 10px 0; color: #6c757d; text-align: center; background: #f8f9fa;">
                <em>Queue is empty. Specify a movie/subfolder name (if needed), paste a MediaFire link, and click <strong>Add to Queue</strong>.</em>
            </div>
            '''))
            return
        
        html_rows = ""
        for i, item in enumerate(mf_link_queue, 1):
            badge_color = {
                "Pending": "#6c757d",
                "Transferring": "#007bff",
                "Completed": "#28a745",
                "Failed": "#dc3545"
            }.get(item["status"], "#6c757d")
            
            subfolder_display = item.get('subfolder_display') or item.get('subfolder', '')
            if subfolder_display:
                folder_tag = f'<span style="background-color: #e8f0fe; color: #1a73e8; border: 1px solid #d2e3fc; padding: 3px 8px; border-radius: 6px; font-weight: 500;">📁 {subfolder_display}</span>'
            else:
                folder_tag = '<span style="color: #adb5bd; font-style: italic;">Root (Base Folder)</span>'
            
            html_rows += f'''
            <tr style="border-bottom: 1px solid #dee2e6;">
                <td style="padding: 8px 12px; font-weight: bold; width: 35px; text-align: center;">{i}</td>
                <td style="padding: 8px 12px; width: 220px;">{folder_tag}</td>
                <td style="padding: 8px 12px; font-family: monospace; word-break: break-all;">{item['link']}</td>
                <td style="padding: 8px 12px; width: 130px; text-align: center;">
                    <span style="background-color: {badge_color}; color: white; padding: 4px 10px; border-radius: 12px; font-size: 12px; font-weight: bold;">
                        {item['status']}
                    </span>
                </td>
            </tr>
            '''
            
        table_html = f'''
        <div style="border: 1px solid #dee2e6; border-radius: 8px; overflow: hidden; margin: 10px 0;">
            <table style="width: 100%; border-collapse: collapse; font-size: 13px; text-align: left;">
                <thead>
                    <tr style="background-color: #f1f3f4; border-bottom: 2px solid #dee2e6;">
                        <th style="padding: 10px 12px; text-align: center;">#</th>
                        <th style="padding: 10px 12px;">Subfolder / Movie</th>
                        <th style="padding: 10px 12px;">MediaFire Link</th>
                        <th style="padding: 10px 12px; text-align: center;">Status</th>
                    </tr>
                </thead>
                <tbody>
                    {html_rows}
                </tbody>
            </table>
        </div>
        '''
        display(HTML(table_html))

def on_add_clicked(b=None):
    raw_text = link_input.value.strip()
    movie_val = subfolder_input.value.strip()
    if not movie_val and folder_dropdown.value and not folder_dropdown.value.startswith('['):
        movie_val = folder_dropdown.value

    # Auto-fallback: if no movie folder is specified, auto-pull from link
    if not movie_val and raw_text:
        first_link = raw_text.splitlines()[0].strip() if '\n' in raw_text else raw_text.split(',')[0].strip()
        auto_title = get_direct_or_mediafire_title(first_link)
        if auto_title:
            movie_val = auto_title
            subfolder_input.value = auto_title

    inner_val = inner_subfolder_input.value.strip()
    if not inner_val and inner_dropdown.value and not inner_dropdown.value.startswith('['):
        inner_val = inner_dropdown.value

    subfolder, display_name = resolve_subfolder_path(movie_val, inner_val)
    
    if not raw_text:
        with status_output:
            clear_output()
            print("\033[91m[ERROR] Please enter a valid MediaFire link!\033[0m")
        return
        
    links = extract_mediafire_links(raw_text)
    if not links:
        candidates = [l.strip() for l in raw_text.split() if l.strip()]
        links = candidates
        
    added_count = 0
    updated_count = 0
    for link in links:
        existing = next((item for item in mf_link_queue if item["link"] == link), None)
        if existing:
            existing["subfolder"] = subfolder
            existing["subfolder_display"] = display_name
            existing["status"] = "Pending"
            updated_count += 1
        else:
            mf_link_queue.append({"link": link, "subfolder": subfolder, "subfolder_display": display_name, "status": "Pending"})
            added_count += 1
            
    link_input.value = ""
    with status_output:
        clear_output()
        parts = []
        if added_count > 0:
            parts.append(f"Added {added_count} new link(s)")
        if updated_count > 0:
            parts.append(f"Reset {updated_count} existing link(s) to 'Pending'")
        folder_info = f" in folder '{display_name}'" if display_name else " in Base folder"
        print(f"\033[92m[INFO] {', '.join(parts)}{folder_info}.\033[0m")
    render_queue()

def on_remove_clicked(b):
    if mf_link_queue:
        removed = mf_link_queue.pop()
        with status_output:
            clear_output()
            folder_info = f" ({removed.get('subfolder')})" if removed.get('subfolder') else ""
            print(f"\033[93m[INFO] Removed last item: {removed['link']}{folder_info}\033[0m")
        render_queue()
    else:
        with status_output:
            clear_output()
            print("\033[93m[INFO] Queue is already empty.\033[0m")

def on_clear_clicked(b):
    global mf_link_queue
    mf_link_queue = []
    with status_output:
        clear_output()
        print("\033[93m[INFO] Queue cleared.\033[0m")
    render_queue()

def on_reset_clicked(b):
    for item in mf_link_queue:
        item["status"] = "Pending"
    with status_output:
        clear_output()
        print("\033[92m[INFO] All items in queue have been reset to 'Pending'. Ready to transfer!\033[0m")
    render_queue()

def run_transfers(b):
    if not mf_link_queue:
        with status_output:
            clear_output()
            print("\033[91m[ERROR] The queue is empty! Add at least one link before starting.\033[0m")
        return
        
    gdrive_root = get_gdrive_root()
    if not gdrive_root:
        with status_output:
            clear_output()
            print("\033[93m[NOTICE] Google Drive not mounted yet. Attempting automatic mount...\033[0m")
        try:
            from google.colab import drive
            drive.mount('/content/drive')
            gdrive_root = get_gdrive_root()
        except:
            pass
            
    if not gdrive_root:
        with status_output:
            clear_output()
            print("\033[91m" + "="*60)
            print("[CRITICAL ERROR] GOOGLE DRIVE IS NOT CONNECTED!")
            print("="*60 + "\033[0m")
            print("\033[93mPlease run 'Step 1: Connect your Google Drive' cell above and authorize your account.\033[0m")
            print("\033[93mWithout Step 1, files cannot be saved to Google Drive!\033[0m")
        return
    
    if all(item["status"] in ("Completed", "Failed") for item in mf_link_queue):
        for item in mf_link_queue:
            item["status"] = "Pending"
        render_queue()

    add_btn.disabled = True
    remove_btn.disabled = True
    clear_btn.disabled = True
    reset_btn.disabled = True
    start_btn.disabled = True
    link_input.disabled = True
    subfolder_input.disabled = True
    base_folder_input.disabled = True
    folder_dropdown.disabled = True
    refresh_folders_btn.disabled = True
    inner_dropdown.disabled = True
    inner_subfolder_input.disabled = True
    
    base_gdrive = sanitize_folder_name(base_folder_input.value.strip()) or "MediaFire_Transfers"
    base_target_dir = os.path.join(gdrive_root, base_gdrive)
    os.makedirs(base_target_dir, exist_ok=True)
    
    with status_output:
        clear_output()
        print(f"\033[94m[INFO] Google Drive Destination: {base_target_dir}\033[0m")
        print(f"\033[94m[INFO] Total links queued: {len(mf_link_queue)}\033[0m")
        print("\033[1m[NOTE] Live download progress for each link will stream below:\033[0m\n")
        
        successful = 0
        failed = []
        
        for idx, item in enumerate(mf_link_queue, 1):
            if item["status"] == "Completed":
                continue
                
            item["status"] = "Transferring"
            render_queue()
            
            page_url = item["link"]
            subfolder = item.get("subfolder", "").strip()
            if subfolder:
                target_dir = os.path.join(base_target_dir, subfolder)
                dest_display = f"{base_gdrive} / {subfolder}"
            else:
                target_dir = base_target_dir
                dest_display = base_gdrive
                
            os.makedirs(target_dir, exist_ok=True)
            
            print(f"\n\033[95m{'='*60}\033[0m")
            print(f"\033[94m[TRANSFER {idx}/{len(mf_link_queue)}]\033[0m")
            print(f"\033[96m[DESTINATION]\033[0m Google Drive -> {dest_display}")
            print(f"\033[96m[DRIVE PATH]\033[0m {target_dir}")
            print(f"\033[96m[PAGE URL]\033[0m {page_url}")
            print("\033[93m[RESOLVING]\033[0m Extracting high-speed direct download link...")
            
            direct_url = get_mediafire_direct_url(page_url)
            if not direct_url:
                print(f"\033[91m[ERROR] Failed to extract direct download link from {page_url}\033[0m")
                item["status"] = "Failed"
                failed.append((page_url, dest_display))
                render_queue()
                continue
                
            parsed_path = urllib.parse.urlparse(direct_url).path
            raw_filename = os.path.basename(parsed_path) or "downloaded_file"
            raw_filename = urllib.parse.unquote(raw_filename)
            
            out_filename = get_unique_filename(target_dir, raw_filename)
            if out_filename != raw_filename:
                print(f"\033[93m[DUPLICATE DETECTED] '{raw_filename}' already exists in folder! Saving as: '{out_filename}'\033[0m")
                
            print("\033[92m[DIRECT URL FOUND]\033[0m Starting 16x multi-connection aria2 stream directly to Google Drive...")
            print(f"\033[95m{'='*60}\033[0m")
            
            master, slave = pty.openpty()
            cmd = [
                "aria2c",
                "-x", "16",
                "-s", "16",
                "-k", "1M",
                "--file-allocation=none",
                "--summary-interval=1",
                "-d", target_dir,
                "-o", out_filename,
                direct_url
            ]
            
            process = subprocess.Popen(cmd, stdout=slave, stderr=slave, text=True, close_fds=True)
            os.close(slave)
            
            fl = fcntl.fcntl(master, fcntl.F_GETFL)
            fcntl.fcntl(master, fcntl.F_SETFL, fl | os.O_NONBLOCK)
            
            while True:
                try:
                    chunk = os.read(master, 1024)
                    if chunk:
                        sys.stdout.write(chunk.decode('utf-8', errors='ignore'))
                        sys.stdout.flush()
                except OSError:
                    pass
                
                if process.poll() is not None:
                    try:
                        remaining = os.read(master, 4096)
                        if remaining:
                            sys.stdout.write(remaining.decode('utf-8', errors='ignore'))
                            sys.stdout.flush()
                    except:
                        pass
                    break
                time.sleep(0.1)
                
            try:
                os.close(master)
            except:
                pass
                
            if process.poll() is None:
                process.terminate()
                process.wait()
                
            if process.returncode == 0:
                try:
                    os.sync()
                except:
                    pass
                    
                item["status"] = "Completed"
                successful += 1
                
                fp = os.path.join(target_dir, out_filename)
                try:
                    f_size = os.path.getsize(fp) / (1024 * 1024)
                    print(f"\n\033[92m[SUCCESS] Completed transfer {idx}/{len(mf_link_queue)} directly into '{dest_display}'!\033[0m")
                    print(f"\033[92m[VERIFIED ON DRIVE] 📄 {out_filename} ({f_size:.2f} MB)\033[0m")
                except:
                    print(f"\n\033[92m[SUCCESS] Completed transfer {idx}/{len(mf_link_queue)} -> {dest_display}/{out_filename}\033[0m")
            else:
                item["status"] = "Failed"
                failed.append((page_url, dest_display))
                print(f"\n\033[91m[FAILED] Link {idx} exited with error code {process.returncode}\033[0m")
            
            render_queue()
        
        try:
            os.sync()
        except:
            pass
            
        print(f"\n\033[95m{'='*60}\033[0m")
        print(f"\033[92m[SUMMARY] {successful}/{len(mf_link_queue)} link(s) transferred successfully!\033[0m")
        if failed:
            print("\033[91m[FAILED ITEMS]:\033[0m")
            for fl, d in failed:
                print(f"  - {fl} ({d})")
                
        print(f"\n\033[96m[GOOGLE DRIVE WEB SYNC TIP]:\033[0m")
        print("1. If you don't see the folder on https://drive.google.com immediately:")
        print("   - Press 'Shift + R' or 'F5' in your browser tab to refresh Google Drive.")
        print("2. Check your Google Account avatar in the top-right corner of Google Drive:")
        print("   - Make sure you are viewing the EXACT SAME Google Account that was authorized in Step 1!")
        print(f"\033[95m{'='*60}\033[0m")

    add_btn.disabled = False
    remove_btn.disabled = False
    clear_btn.disabled = False
    reset_btn.disabled = False
    start_btn.disabled = False
    link_input.disabled = False
    subfolder_input.disabled = False
    base_folder_input.disabled = False
    folder_dropdown.disabled = False
    refresh_folders_btn.disabled = False
    inner_dropdown.disabled = False
    inner_subfolder_input.disabled = False
    folders = get_existing_gdrive_folders()
    folder_dropdown.options = ["[ ➕ New Movie / Type Below ]"] + folders

add_btn.on_click(on_add_clicked)
link_input.on_submit(on_add_clicked)
remove_btn.on_click(on_remove_clicked)
clear_btn.on_click(on_clear_clicked)
reset_btn.on_click(on_reset_clicked)
start_btn.on_click(run_transfers)

buttons_row = widgets.HBox([add_btn, remove_btn, clear_btn, reset_btn], layout=widgets.Layout(margin='6px 0'))
start_row = widgets.HBox([start_btn], layout=widgets.Layout(margin='10px 0'))

step1_header = widgets.HTML("<div style='font-size: 13px; font-weight: 700; color: #1a73e8; margin: 4px 0 2px 0;'>🔗 Step 1: MediaFire / Direct Web Link & Auto-Name</div>")
step2_header = widgets.HTML("<div style='font-size: 13px; font-weight: 700; color: #1a73e8; margin: 10px 0 2px 0;'>📁 Step 2: Google Drive Destination (Movie & Inside Subfolders)</div>")
step3_header = widgets.HTML("<div style='font-size: 12px; font-weight: 600; color: #70757a; margin: 8px 0 2px 0;'>⚙️ Base Google Drive Folder:</div>")

display(widgets.VBox([
    widgets.HTML("<h3 style='margin: 0 0 10px 0; color: #1a73e8;'>⚡ MediaFire & Direct Web Links to Google Drive Accelerated Queue</h3>"),
    step1_header,
    link_row,
    step2_header,
    folder_select_row,
    subfolder_input,
    inner_dropdown,
    inner_subfolder_input,
    step3_header,
    base_folder_input,
    buttons_row,
    start_row,
    queue_output,
    status_output
]))

render_queue()


### Step 3.6: Instant Google Drive Explorer & Sync
Run this cell to immediately inspect all downloaded files and folders in your Google Drive without opening the Drive web UI.

In [ ]:
#@title Check Transferred Files in Google Drive
import os

try:
    os.sync()
except:
    pass

def get_gdrive_root():
    for candidate in ["/content/drive/MyDrive", "/content/drive/My Drive"]:
        if os.path.exists(candidate):
            return candidate
    return None

gdrive_root = get_gdrive_root()

if not gdrive_root:
    print("\033[91m[ERROR] Google Drive is NOT mounted! Please run Step 1 first.\033[0m")
else:
    print(f"\033[92m[CONNECTED] Google Drive Root: {gdrive_root}\033[0m")
    
    base_folder_name = "MediaFire_Transfers"
    if 'base_folder_input' in globals() and base_folder_input.value.strip():
        base_folder_name = base_folder_input.value.strip()
        
    target_path = os.path.join(gdrive_root, base_folder_name)
    
    if not os.path.exists(target_path):
        print(f"\n\033[93m[NOTICE] Folder '{base_folder_name}' not found yet in {gdrive_root}.\033[0m")
        print("\033[93mAvailable folders in your Google Drive root:\033[0m")
        for item in sorted(os.listdir(gdrive_root)):
            if os.path.isdir(os.path.join(gdrive_root, item)) and not item.startswith('.'):
                print(f"  📁 {item}")
    else:
        print(f"\n\033[92m[FOUND] Google Drive Folder: {target_path}\033[0m\n")
        total_files = 0
        for root, dirs, files in os.walk(target_path):
            level = root.replace(target_path, '').count(os.sep)
            indent = ' ' * 4 * level
            folder_name = os.path.basename(root)
            print(f"{indent}📁 \033[1m{folder_name}/\033[0m")
            subindent = ' ' * 4 * (level + 1)
            for f in sorted(files):
                if f.startswith('.'):
                    continue
                fp = os.path.join(root, f)
                try:
                    size_mb = os.path.getsize(fp) / (1024 * 1024)
                    print(f"{subindent}📄 {f} ({size_mb:.2f} MB)")
                except:
                    print(f"{subindent}📄 {f}")
                total_files += 1
                
        if total_files == 0:
            print("\n(Folder exists, but no files have been downloaded yet)")
        else:
            print(f"\n\033[94m[INFO] Total files verified in Google Drive: {total_files}\033[0m")
            print("\n\033[96m[TIP] If not visible on drive.google.com in browser:\033[0m")
            print("1. Refresh the Google Drive tab with F5 or Shift + R.")
            print("2. Ensure your browser is viewing the SAME Google Account authorized in Step 1.")


### Step 4: Force Sync & Disconnect (Optional)
Run this cell once all your transfers are complete to instantly flush all cached data to Google Drive servers (so they appear on your phone/browser immediately) and safely unmount the connection.

In [ ]:
from google.colab import drive
print("[INFO] Flushing cache and disconnecting Drive...")
drive.flush_and_unmount()
print("[SUCCESS] Google Drive successfully synced and unmounted!")